# Crear y Compilar la Red MS-CLSTM (Proto)

Este notebook define y compila la arquitectura temporal MS-CLSTM.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.layers import Layer
import os

# Función puramente robusta para buscar y cargar el archivo .env
def load_env_variables():
    import os
    from pathlib import Path
    
    # 1. Buscar .env subiendo niveles desde el CWD actual
    try:
        start_dir = Path(os.getcwd())
    except:
        start_dir = Path(".")
        
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
            
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
        
    # 2. Leer e inyectar variables en os.environ
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
            
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()

models_dir = os.environ["MODELS_DL_PROTO"]
os.makedirs(models_dir, exist_ok=True)

✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


In [ ]:
@tf.keras.utils.register_keras_serializable()
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1), initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1), initializer="zeros", trainable=True)
        super(AttentionLayer, self).build(input_shape)
    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return tf.keras.backend.sum(output, axis=1)
    def get_config(self): 
        return super(AttentionLayer, self).get_config()

In [ ]:
def residual_inception_block(input_tensor, filters):
    branch1 = layers.Conv1D(filters, 1, padding='same', activation='relu')(input_tensor)
    branch3 = layers.Conv1D(filters, 3, padding='same', activation='relu')(input_tensor)
    branch5 = layers.Conv1D(filters, 5, padding='same', activation='relu')(input_tensor)
    concat = layers.concatenate([branch1, branch3, branch5], axis=-1)
    projection = layers.Conv1D(filters * 3, 1, padding='same')(input_tensor)
    x = layers.add([concat, projection])
    x = layers.Activation('relu')(x)
    x = layers.BatchNormalization()(x)
    return x

def build_myotensor_proto_model(input_shape=(300, 1), num_classes=4):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)
    x = residual_inception_block(x, 64)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = AttentionLayer()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs=inputs, outputs=outputs, name="MS_CLSTM_Proto")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
model = build_myotensor_proto_model((300, 1), 4)
model_path = os.path.join(models_dir, "myotensor_proto_net.keras")
print(f"💾 Guardando modelo inicial en: {model_path} ...")
model.save(model_path)
print("¡Arquitectura MS-CLSTM guardada con éxito!")

💾 Guardando modelo inicial en: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/models/myotensor_proto/dl/myotensor_proto_net.keras ...
🎉 ¡Arquitectura MS-CLSTM guardada con éxito!
